# 02 - Limpieza y normalización de licitaciones

Este notebook recorre el paso de las respuestas crudas de la API (un JSON por licitación) a una tabla lista para analizar, y revisa la calidad de los datos resultantes.

**Requisito:** necesita los JSON de detalle en `data/raw/details/`, que no se versionan porque incluyen datos de funcionarios. Para obtenerlos hace falta un ticket de la API y ejecutar `python -m app.cli build-bulk-dataset` (ver el README). El notebook `03_eda_oportunidades.ipynb` no tiene este requisito: trabaja con la muestra pública que se genera al final de este.

Las salidas guardadas corresponden a una corrida de 1.407 licitaciones descargadas el 5 y 6 de abril de 2026. En este notebook no se imprime ningún campo personal.

In [1]:
import json
import sys
from pathlib import Path

import pandas as pd

sys.path.insert(0, "..")

from app.services.sample import SAMPLE_COLUMNS, build_public_sample
from app.services.transform import normalize_licitacion_detail, normalize_licitacion_items

DETAILS_DIR = Path("../data/raw/details")
SAMPLE_PATH = Path("../data/sample/licitaciones_muestra.csv")

files = sorted(DETAILS_DIR.glob("*.json"))
if not files:
    raise FileNotFoundError("No hay JSON en data/raw/details/. Revisa el requisito al inicio del notebook.")
print(f"Archivos de detalle: {len(files)}")

Archivos de detalle: 1407


## 1. Estructura de una respuesta cruda

Cada archivo es la respuesta de la API para un código de licitación. Los datos vienen anidados: la licitación está dentro de `Listado`, y a su vez contiene bloques como `Comprador`, `Fechas` e `Items`. Se muestran solo los nombres de los campos, no sus valores.

In [2]:
payload = json.loads(files[0].read_text(encoding="utf-8"))
licitacion = payload["Listado"][0]

print("Nivel superior:", list(payload))
print(f"Campos de la licitación: {len(licitacion)}")
for bloque in ("Comprador", "Fechas", "Items"):
    print(f"  {bloque}: {list(licitacion[bloque])}")

Nivel superior: ['Cantidad', 'FechaCreacion', 'Version', 'Listado']
Campos de la licitación: 54
  Comprador: ['CodigoOrganismo', 'NombreOrganismo', 'RutUnidad', 'CodigoUnidad', 'NombreUnidad', 'DireccionUnidad', 'ComunaUnidad', 'RegionUnidad', 'RutUsuario', 'CodigoUsuario', 'NombreUsuario', 'CargoUsuario']
  Fechas: ['FechaCreacion', 'FechaCierre', 'FechaInicio', 'FechaFinal', 'FechaPubRespuestas', 'FechaActoAperturaTecnica', 'FechaActoAperturaEconomica', 'FechaPublicacion', 'FechaAdjudicacion', 'FechaEstimadaAdjudicacion', 'FechaSoporteFisico', 'FechaTiempoEvaluacion', 'FechaEstimadaFirma', 'FechasUsuario', 'FechaVisitaTerreno', 'FechaEntregaAntecedentes']
  Items: ['Cantidad', 'Listado']


## 2. Normalización

`normalize_licitacion_detail` aplana cada respuesta a una fila: limpia espacios, convierte fechas, números y booleanos, y tolera campos ausentes. `normalize_licitacion_items` hace lo mismo con los ítems, una fila por ítem.

In [3]:
payloads = {path.stem: json.loads(path.read_text(encoding="utf-8")) for path in files}

details = pd.concat(
    [
        normalize_licitacion_detail(data, code_override=code).dropna(axis=1, how="all")
        for code, data in payloads.items()
    ],
    ignore_index=True,
)
items = pd.concat(
    [normalize_licitacion_items(data).dropna(axis=1, how="all") for data in payloads.values()],
    ignore_index=True,
)

print(f"Licitaciones: {details.shape[0]} filas, {details.shape[1]} columnas")
print(f"Ítems:        {items.shape[0]} filas, {items.shape[1]} columnas")
print(f"Ítems por licitación: mediana {items.groupby('codigo_externo').size().median():.0f}, máximo {items.groupby('codigo_externo').size().max()}")

Licitaciones: 1407 filas, 60 columnas
Ítems:        6172 filas, 12 columnas
Ítems por licitación: mediana 1, máximo 190


## 3. Revisión de calidad

### 3.1 Duplicados y estado de la extracción

In [4]:
print("Códigos duplicados:", int(details["codigo_externo"].duplicated().sum()))
print("Estado de extracción:", details["extraction_status"].value_counts().to_dict())

Códigos duplicados: 0
Estado de extracción: {'success': 1407}


### 3.2 Completitud de los campos de análisis

Porcentaje de licitaciones con valor en cada campo que se usa después.

In [5]:
analysis_columns = [
    "nombre", "descripcion", "estado", "tipo", "fecha_publicacion", "fecha_cierre",
    "organismo_nombre", "region_unidad", "comuna_unidad", "moneda", "monto_estimado",
]
completeness = (details[analysis_columns].notna().mean() * 100).round(1).rename("% con valor").to_frame()
completeness

,% con valor
nombre,100.0
descripcion,100.0
estado,100.0
tipo,100.0
fecha_publicacion,100.0
fecha_cierre,100.0
organismo_nombre,100.0
region_unidad,100.0
comuna_unidad,92.3
moneda,100.0


### 3.3 Estados

La API entrega un código y un nombre de estado. Se comprueba que cada código corresponda a un único nombre.

In [6]:
states = details.groupby(["codigo_estado", "estado"]).size().rename("licitaciones").reset_index()
assert states["codigo_estado"].is_unique
states

,codigo_estado,estado,licitaciones
0,5,Publicada,498
1,6,Cerrada,347
2,7,Desierta (o art. 3 ó 9 Ley 19.886),81
3,8,Adjudicada,450
4,15,Revocada,29
5,16,Suspendida,2


### 3.4 Montos y monedas

El monto estimado no es obligatorio y puede venir en distintas monedas, por lo que no se pueden sumar sin separar.

In [7]:
amount = details["monto_estimado"]
print(f"Con monto estimado: {amount.notna().sum()} de {len(details)} ({amount.notna().mean():.1%})")
print("Moneda (todas las licitaciones):", details["moneda"].value_counts().to_dict())
print("Moneda (solo con monto):       ", details.loc[amount.notna(), "moneda"].value_counts().to_dict())

clp = details[(details["moneda"] == "CLP") & amount.notna()]
print(f"\nMontos en CLP: {len(clp)}")
print(f"  menores a $1.000 (probables valores de relleno): {int((clp['monto_estimado'] < 1000).sum())}")
print(f"  mediana: ${clp['monto_estimado'].median():,.0f}   máximo: ${clp['monto_estimado'].max():,.0f}")

Con monto estimado: 696 de 1407 (49.5%)
Moneda (todas las licitaciones): {'CLP': 1381, 'CLF': 22, 'USD': 3, 'UTM': 1}
Moneda (solo con monto):        {'CLP': 681, 'CLF': 12, 'USD': 2, 'UTM': 1}

Montos en CLP: 681
  menores a $1.000 (probables valores de relleno): 3
  mediana: $24,500,000   máximo: $70,157,421,152


### 3.5 Fechas

In [8]:
published = pd.to_datetime(details["fecha_publicacion"])
closing = pd.to_datetime(details["fecha_cierre"])

print(f"Publicación: {published.min():%Y-%m-%d} a {published.max():%Y-%m-%d}")
print(f"Cierre:      {closing.min():%Y-%m-%d} a {closing.max():%Y-%m-%d}")
print("Cierre anterior a la publicación:", int((closing < published).sum()))
print("Publicadas en marzo o abril de 2026:", int((published >= "2026-03-01").sum()), f"({(published >= '2026-03-01').mean():.1%})")

Publicación: 2024-07-17 a 2026-04-02
Cierre:      2024-10-14 a 2026-05-22
Cierre anterior a la publicación: 0
Publicadas en marzo o abril de 2026: 1062 (75.5%)


**Lectura.**

- La extracción es limpia: 1.407 licitaciones, sin códigos duplicados ni descargas fallidas, y los campos de identificación, estado, fechas, organismo y región están completos.
- **El monto estimado es el punto débil: solo el 49,5 % de las licitaciones lo informa.** Cualquier análisis de montos describe a la mitad de la muestra, no al total.
- Hay cuatro monedas. El 97,8 % de los montos informados está en pesos (681 de 696); los 15 restantes (UF, dólares y UTM) se dejan fuera de los análisis de monto para no mezclar unidades.
- Tres montos en pesos son menores a $1.000 (dos valen $1): son valores de relleno, no estimaciones reales.
- Los montos son muy dispersos: la mediana es de $24,5 millones y el máximo supera los $70.000 millones. Conviene resumirlos con medianas y no con promedios.
- La muestra no es un periodo completo: reúne lo que la API devolvió al consultar unos pocos días de abril de 2026. El 75,5 % de las licitaciones se publicó en marzo o abril de 2026, pero hay algunas publicadas desde julio de 2024 que seguían apareciendo. No sirve para analizar tendencias en el tiempo.

## 4. Muestra pública sin datos personales

Las respuestas incluyen campos con datos de personas. Se listan los nombres de esas columnas (no su contenido) y cuántas licitaciones los traen.

In [9]:
personal_columns = [
    column
    for column in details.columns
    if any(token in column for token in ("usuario", "responsable", "direccion"))
]
(details[personal_columns].notna().sum().rename("licitaciones con valor").to_frame())

,licitaciones con valor
direccion_unidad,1306
codigo_usuario,1407
nombre_usuario,1407
cargo_usuario,1338
nombre_responsable_pago,1247
nombre_responsable_contrato,1247


`build_public_sample` genera la tabla que sí se versiona: conserva los campos de análisis, agrega el rubro principal (la categoría de primer nivel más frecuente entre los ítems de la licitación) y deja fuera los campos anteriores y la descripción en texto libre, que a veces nombra personas de contacto.

In [10]:
sample = build_public_sample(DETAILS_DIR)

assert list(sample.columns) == SAMPLE_COLUMNS
assert not set(personal_columns) & set(sample.columns)
assert "descripcion" not in sample.columns

SAMPLE_PATH.parent.mkdir(parents=True, exist_ok=True)
sample.to_csv(SAMPLE_PATH, index=False, encoding="utf-8")

print(f"Muestra: {sample.shape[0]} filas, {sample.shape[1]} columnas, {SAMPLE_PATH.stat().st_size / 1024:.0f} KB")
print("Columnas:", list(sample.columns))
print(f"Licitaciones sin rubro principal: {int(sample['rubro_principal'].isna().sum())}")

Muestra: 1407 filas, 14 columnas, 377 KB
Columnas: ['codigo_externo', 'nombre', 'codigo_estado', 'estado', 'tipo', 'fecha_publicacion', 'fecha_cierre', 'organismo_nombre', 'comuna_unidad', 'region_unidad', 'moneda', 'monto_estimado', 'rubro_principal', 'cantidad_items']
Licitaciones sin rubro principal: 0
